# syft-bg Integration Tests

Reusable notebook that tests `syft_bg` service management:

1. **Config & Status basics** — init, inspect config, check status
2. **Misconfigured `email_approve`** — missing `gcp_project_id` → graceful error
3. **Start / Stop / Restart** — verify PID lifecycle
4. **Auto-approve job via API** — `auto_approve_job()`, follow-up auto-approval, list/remove rules
5. **Job repr for auto-approved jobs (DS side)** — verify approval_method shows `auto`
6. **Invalid DO token at startup** — service reports ERROR status

### Prerequisites
- `.env` file with `DO_EMAIL`, `DS_EMAIL`, `GCP_PROJECT_ID`, `APP_CREDENTIALS`, `TOKEN_DS`, `TOKEN_DO_WITH_SCOPES`
- OAuth credential files (see Setup cells for first-time token generation)

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup: Environment & Credentials

In [ ]:
import os
from pathlib import Path

for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]
GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]

APP_CREDENTIALS = Path(os.environ["APP_CREDENTIALS"])
do_token_path = Path(os.environ["TOKEN_DO_WITH_SCOPES"])
ds_token_path = Path(os.environ["TOKEN_DS"])


assert do_token_path.exists() and ds_token_path.exists()


### First-time token generation

Uncomment and run these cells once to generate OAuth tokens. Each opens a browser for consent.

In [ ]:
import syft_client as sc

# --- DS token (Drive scopes only) ---
# sc.credentials_to_token(
#     credentials_path=str(APP_CREDENTIALS),
#     output_path=str(TOKEN_DS),
# )
# print(f"DS token saved to: {TOKEN_DS}")

In [ ]:
# --- DO token (Drive + Gmail + PubSub scopes) ---
# sc.credentials_to_token(
#     credentials_path=str(DO_CREDENTIALS),
#     output_path=str(DO_TOKEN),
#     do_scopes=True,
# )
# print(f"DO token saved to: {DO_TOKEN}")

In [ ]:
assert ds_token_path.exists(), f"DS token missing: {ds_token_path} — uncomment generation cell above"
assert do_token_path.exists(), f"DO token missing: {do_token_path} — uncomment generation cell above"
print("All tokens ready")

## Setup: Login & Peers

Potentially reset

In [ ]:
# import syft_bg
# syft_bg.reset()
# sc.delete_syftbox(token_path=do_token_path, email=DO_EMAIL)
# sc.delete_syftbox(token_path=ds_token_path, email=DS_EMAIL)

In [ ]:
import syft_client as sc

do_client = sc.login_do(email=DO_EMAIL, token_path=do_token_path)
ds_client = sc.login_ds(email=DS_EMAIL, token_path=ds_token_path)

In [ ]:
ds_client.add_peer(DO_EMAIL)
do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)
print("Peer relationship established")

## Setup: Test Dataset

In [ ]:
try:
    do_client.create_dataset(
        name="testdata",
        mock_path="data/mock.txt",
        private_path="data/private.txt",
        users=[DS_EMAIL],
    )
except FileExistsError:
    print("Dataset 'testdata' already exists, skipping creation")

---
## Test 1: Config & Status Basics

In [ ]:
import syft_bg

syft_bg.reset()

In [ ]:
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=do_token_path,
    settings={
        "email_approve": {"gcp_project_id": GCP_PROJECT_ID},
    },
)

In [ ]:
# Inspect config — should show do_email, token paths, and gcp_project_id
# syft_bg.config

In [ ]:
# Status before starting any services — all should be stopped
status = syft_bg.status

status

In [ ]:
for name, info in status.service_infos.items():
    assert info.status.value in ("stopped", "unknown"), f"{name} should be stopped, got {info.status}"
print("\nAll services stopped as expected")

---
## Test 2: Misconfigured `email_approve` (no `gcp_project_id`)

In [ ]:
from syft_bg.services.base import ServiceStatus
import time

In [ ]:
syft_bg.reset()
syft_bg.init(do_email=DO_EMAIL, token_path=do_token_path)

syft_bg.ensure_running(services=["email_approve"])
time.sleep(3)  # wait for subprocess to fail

ea_info = syft_bg.status.service_infos["email_approve"]
assert ea_info.status == ServiceStatus.ERROR, f"Expected ERROR, got {ea_info.status}"
assert ea_info.error and "gcp_project_id" in ea_info.error
print(f"email_approve: {ea_info.status.value}")
print(f"Error confirms missing gcp_project_id")

---
## Test 3: Start / Stop / Restart Services

In [ ]:
syft_bg.reset()
syft_bg.init(do_email=DO_EMAIL, token_path=do_token_path,
             settings={"email_approve": {"gcp_project_id": GCP_PROJECT_ID}})

SERVICES = ["sync", "notify", "approve"]
syft_bg.ensure_running(services=["sync"], restart=True)
syft_bg.ensure_running(services=["notify", "approve"], restart=True)

In [ ]:
status = syft_bg.status
initial_pids = {}
for svc in SERVICES:
    info = status.service_infos[svc]
    assert info.status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), f"{svc}: {info.status}"
    initial_pids[svc] = info.pid
    print(f"{svc}: PID {info.pid}")
print(f"All {len(SERVICES)} services running")

In [ ]:
# Record initial PIDs
initial_pids = {}
for svc in SERVICES:
    info = status.service_infos[svc]
    assert info.status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), (
        f"{svc} should be running, got {info.status}"
    )
    initial_pids[svc] = info.pid
    print(f"{svc}: PID {info.pid} ({info.status.value})")

print(f"\nAll {len(SERVICES)} services running")

In [ ]:
# Restart approve — PID should change
syft_bg.restart("approve")
new_pid = syft_bg.status.service_infos["approve"].pid
assert new_pid != initial_pids["approve"], f"PID didn't change: {new_pid}"
print(f"approve restarted: {initial_pids['approve']} -> {new_pid}")
     

In [ ]:
# Stop approve, others keep running
syft_bg.stop("approve")
s = syft_bg.status
assert s.service_infos["approve"].status == ServiceStatus.STOPPED
for svc in ["sync", "notify"]:
    assert s.service_infos[svc].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING)
print("approve stopped, sync/notify still running")

In [ ]:

# Start approve again
syft_bg.start("approve")
assert syft_bg.status.service_infos["approve"].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING)
print("Start/Stop/Restart test passed")

---
## Test 4: Auto-Approve Job via API

DS submits a parameterized job. DO uses `syft_bg.auto_approve_job()` to create an
auto-approval rule. The `approve` service picks it up, runs the job, and auto-approves
a follow-up job with the same code but different params.

In [ ]:
import syft_bg
import time
import json
import tempfile
import uuid

def create_parameterized_job(main_file: str, params: dict) -> Path:
    """Create a temp project dir with a script and params.json."""
    project_dir = Path(tempfile.mkdtemp(prefix="syftbg_test_"))
    (project_dir / main_file).write_text(
        'import os, json\n'
        '\n'
        'with open("params.json") as f:\n'
        '    params = json.load(f)\n'
        '\n'
        'os.makedirs("outputs", exist_ok=True)\n'
        'with open("outputs/result.json", "w") as f:\n'
        '    json.dump({"params": params, "status": "done"}, f)\n'
        '\n'
        'print(f"Result: {params}")\n'
    )
    (project_dir / "params.json").write_text(json.dumps(params))
    return project_dir

In [ ]:
# Ensure services are running for this test
syft_bg.reset()
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=do_token_path,
    settings={
        "email_approve": {"gcp_project_id": GCP_PROJECT_ID},
    },
)
syft_bg.ensure_running(services=["sync", "notify", "approve"], restart=True)
time.sleep(5)

In [ ]:
syft_bg.status

### 4a: Submit a job and create auto-approval rule

In [ ]:
JOB_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
job_dir = create_parameterized_job(main_file="main.py", params={"run": 1})

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(job_dir),
    job_name=JOB_NAME,
    entrypoint="main.py",
    force_submission=True,
)
print(f"Submitted: {JOB_NAME}")

In [ ]:
# Wait for job to appear on DO side (sync service keeps local state updated)
pending_job = None
for _ in range(24):
    pending_job = next((j for j in do_client.job_client.jobs if j.name == JOB_NAME), None)
    if pending_job:
        break
    time.sleep(5)

assert pending_job, f"DO never saw job {JOB_NAME}"
print(f"DO sees job: {JOB_NAME} ({pending_job.status})")

In [ ]:
result = syft_bg.auto_approve_job(pending_job, file_paths=["params.json"])
assert result.success, f"auto_approve_job failed: {result.error}"
print(result)
    

In [ ]:

# Verify rule
rules = syft_bg.list_auto_approvals()
assert JOB_NAME in rules
rule = rules[JOB_NAME]
assert {e.relative_path for e in rule.file_contents} == {"main.py"}
assert "params.json" in rule.file_paths
assert DS_EMAIL in rule.peers
assert JOB_NAME in syft_bg.status.auto_approvals
print(f"Rule '{JOB_NAME}' created and visible in status")

In [ ]:
# Wait for job to complete
for _ in range(24):
    job = next((j for j in do_client.job_client.jobs if j.name == JOB_NAME), None)
    if job and job.status == "done":
        break
    time.sleep(5)

assert job and job.status == "done", f"Job not done: {job.status if job else 'missing'}"
print(f"Job {JOB_NAME}: {job.status}")

In [ ]:
# DS fetches result
ds_job = next((j for j in ds_client.job_client.jobs if j.name == JOB_NAME), None)
assert ds_job and ds_job.status == "done"
result_data = json.loads(ds_job.output_paths[0].read_text())
assert result_data["params"]["run"] == 1
print(f"DS result: {result_data}")

In [ ]:
# Follow-up job — should auto-approve
FOLLOWUP_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
ds_client.submit_python_job(user=DO_EMAIL, code_path=str(create_parameterized_job("main.py", {"run": 2})),
                            job_name=FOLLOWUP_NAME, entrypoint="main.py", force_submission=True)
print(f"Submitted follow-up: {FOLLOWUP_NAME}")

In [ ]:
for _ in range(24):
    job2 = next((j for j in do_client.job_client.jobs if j.name == FOLLOWUP_NAME), None)
    if job2 and job2.status == "done":
        break
    time.sleep(5)

assert job2 and job2.status == "done", f"Follow-up not done: {job2.status if job2 else 'missing'}"
print(f"Follow-up {FOLLOWUP_NAME}: {job2.status} (auto-approved!)")

In [ ]:
ds_client.jobs;
ds_job2 = next((j for j in ds_client.job_client.jobs if j.name == FOLLOWUP_NAME), None)
assert ds_job2 and ds_job2.status == "done"
result_data2 = json.loads(ds_job2.output_paths[0].read_text())
assert result_data2["params"]["run"] == 2
print(f"DS follow-up result: {result_data2}")

---
## Test 5: Job Repr for Auto-Approved Jobs (DS Side)

In [ ]:
ds_client.jobs

# Removal

In [ ]:

# Remove rule
r = syft_bg.remove_auto_approve(JOB_NAME)
assert r.success
assert JOB_NAME not in syft_bg.list_auto_approvals()
print(f"Rule '{JOB_NAME}' removed")

---
## Test 6: Invalid DO Token at Startup

In [ ]:
syft_bg.status

In [ ]:

syft_bg.reset()
bad_token = Path(tempfile.mktemp(suffix=".json"))
bad_token.write_text(json.dumps({"invalid": True}))

syft_bg.init(do_email=DO_EMAIL, token_path=str(bad_token),
             settings={"email_approve": {"gcp_project_id": GCP_PROJECT_ID}})

syft_bg.ensure_running(services=["sync"])
time.sleep(3)  # wait for subprocess to fail

sync_info = syft_bg.status.service_infos["sync"]
assert sync_info.status == ServiceStatus.ERROR, f"Expected ERROR, got {sync_info.status}"
assert sync_info.error
print(f"sync: {sync_info.status.value}")
print(f"Error: {sync_info.error[:200]}...")

bad_token.unlink(missing_ok=True)
syft_bg.stop()
print("\nTest 6 passed")

---
## Debugging

In [ ]:
for svc in ("sync", "notify", "approve"):
    print(f"\n{'='*20} {svc} {'='*20}")
    syft_bg.logs(svc, n=20)

In [ ]:
syft_bg.status

---
## Cleanup

In [ ]:
syft_bg.stop()

In [ ]:
# Uncomment to fully reset all syft-bg state
# syft_bg.reset()